# 02 - Feature Engineering
## Construcao de features para o modelo de ranking de acoes
Partimos dos dados brutos do notebook 01 e calculamos:
- Retornos em multiplos horizontes
- Indicadores de volatilidade
- Indicadores tecnicos (RSI, MACD, Bollinger, ATR)
- Features de volume
- Betas rolantes dos fatores Fama-French
- Rankings cross-sectional
- Variavel target (retorno futuro de 21 dias)

In [ ]:
import pandas as pd
import numpy as np
import pandas_datareader.data as web
import warnings
warnings.filterwarnings("ignore")

df = pd.read_parquet("../data/raw/sp500_daily_prices.parquet")
print(f"Shape: {df.shape}")
print(f"Tickers: {df.index.get_level_values('ticker').nunique()}")
df.head()

### 2.1 Retornos logaritmicos
Calculamos retornos log em janelas de 1, 5, 10, 21 e 63 dias uteis.
Retornos log sao aditivos no tempo e tem distribuicao mais proxima da normal.

In [ ]:
horizons = [1, 5, 10, 21, 63]

for h in horizons:
    df[f"ret_{h}d"] = df.groupby("ticker")["close"].transform(
        lambda x: np.log(x / x.shift(h))
    )

print("Retornos calculados:")
print(df.filter(like="ret_").describe().round(4))

### 2.2 Variavel target
O target e o retorno log dos proximos 21 dias uteis (~1 mes).
Usamos shift negativo porque estamos olhando pra frente.

In [ ]:
df["target_21d"] = df.groupby("ticker")["close"].transform(
    lambda x: np.log(x.shift(-21) / x)
)

print(f"Target nulo (ultimos 21 dias de cada ticker): {df['target_21d'].isna().sum()}")
print(f"Target preenchido: {df['target_21d'].notna().sum()}")

### 2.3 Volatilidade
Garman-Klass usa os quatro precos do dia (OHLC) para estimar volatilidade
com mais precisao que o desvio padrao simples do fechamento.
Tambem calculamos o desvio padrao rolante dos retornos diarios.

In [ ]:
gk_var = (
    0.5 * (np.log(df["high"] / df["low"])) ** 2
    - (2 * np.log(2) - 1) * (np.log(df["close"] / df["open"])) ** 2
)
df["volatility_gk_21d"] = gk_var.groupby("ticker").transform(
    lambda x: x.rolling(21).mean()
)
df["volatility_gk_63d"] = gk_var.groupby("ticker").transform(
    lambda x: x.rolling(63).mean()
)

df["volatility_std_21d"] = df.groupby("ticker")["ret_1d"].transform(
    lambda x: x.rolling(21).std()
)
df["volatility_std_63d"] = df.groupby("ticker")["ret_1d"].transform(
    lambda x: x.rolling(63).std()
)

print("Volatilidade:")
print(df.filter(like="volatility").describe().round(6))

### 2.4 Indicadores tecnicos
RSI: mede forca relativa do movimento de preco (0-100).
MACD: diferenca entre medias moveis exponenciais rapida e lenta.
Bollinger Band Width: distancia entre bandas superior e inferior.
ATR: amplitude media do movimento diario.

In [ ]:
def calc_rsi(series, period):
    delta = series.diff()
    gain = delta.where(delta > 0, 0.0)
    loss = -delta.where(delta < 0, 0.0)
    avg_gain = gain.rolling(period).mean()
    avg_loss = loss.rolling(period).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))


def calc_macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line, macd_line - signal_line


def calc_bollinger_width(series, period=20):
    sma = series.rolling(period).mean()
    std = series.rolling(period).std()
    upper = sma + 2 * std
    lower = sma - 2 * std
    return (upper - lower) / sma


def calc_atr(high, low, close, period=14):
    tr1 = high - low
    tr2 = (high - close.shift(1)).abs()
    tr3 = (low - close.shift(1)).abs()
    tr = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
    return tr.rolling(period).mean()


for ticker, group in df.groupby("ticker"):
    idx = group.index

    df.loc[idx, "rsi_14"] = calc_rsi(group["close"], 14)
    df.loc[idx, "rsi_21"] = calc_rsi(group["close"], 21)

    macd_line, signal_line, macd_hist = calc_macd(group["close"])
    df.loc[idx, "macd"] = macd_line
    df.loc[idx, "macd_signal"] = signal_line
    df.loc[idx, "macd_hist"] = macd_hist

    df.loc[idx, "bb_width"] = calc_bollinger_width(group["close"])
    df.loc[idx, "atr_14"] = calc_atr(group["high"], group["low"], group["close"])

print("Indicadores tecnicos:")
print(df[["rsi_14", "rsi_21", "macd_hist", "bb_width", "atr_14"]].describe().round(4))

### 2.5 Features de volume
Dollar volume: preco x volume negociado (proxy de liquidez).
Volume ratio: volume atual vs media movel de 21 dias (detecta picos).
OBV: On Balance Volume, acumula volume em dias de alta e subtrai em dias de baixa.

In [ ]:
df["dollar_volume"] = df["close"] * df["volume"]

df["volume_ratio_21d"] = df.groupby("ticker").apply(
    lambda g: g["volume"] / g["volume"].rolling(21).mean(),
    include_groups=False
).droplevel(0).sort_index()

def calc_obv(group):
    sign = np.sign(group["close"].diff())
    return (sign * group["volume"]).cumsum()

df["obv"] = df.groupby("ticker").apply(
    calc_obv, include_groups=False
).droplevel(0).sort_index()

df["obv_pct_change_21d"] = df.groupby("ticker")["obv"].transform(
    lambda x: x.pct_change(21)
)

print("Volume features:")
print(df[["dollar_volume", "volume_ratio_21d", "obv_pct_change_21d"]].describe().round(4))

### 2.6 Fatores Fama-French
Baixamos os 5 fatores Fama-French do site de Kenneth French via pandas-datareader.
Calculamos betas rolantes (janela de 252 dias) para cada acao em relacao a cada fator.
Os betas indicam a sensibilidade de cada acao a fatores como mercado, tamanho e valor.

In [ ]:
ff_factors = web.DataReader("F-F_Research_Data_5_Factors_2x3_daily", "famafrench", start="2009-01-01")[0]
ff_factors.index = pd.to_datetime(ff_factors.index, format="%Y%m%d")
ff_factors = ff_factors / 100

print(f"Fatores Fama-French: {ff_factors.shape}")
print(f"Periodo: {ff_factors.index.min()} ate {ff_factors.index.max()}")
ff_factors.head()

In [ ]:
factor_cols = ["Mkt-RF", "SMB", "HML", "RMW", "CMA"]
window = 252

daily_returns = df["ret_1d"].unstack("ticker")
daily_returns.index = pd.to_datetime(daily_returns.index)

ff_aligned = ff_factors.reindex(daily_returns.index)

betas = {}
for factor in factor_cols:
    factor_series = ff_aligned[factor]
    beta_name = f"beta_{factor.lower().replace('-', '_')}"

    rolling_cov = daily_returns.rolling(window).cov(factor_series)
    rolling_var = factor_series.rolling(window).var()
    beta = rolling_cov.div(rolling_var, axis=0)

    betas[beta_name] = beta

betas_df = pd.concat(betas, axis=1)
betas_df.columns = betas_df.columns.droplevel(1) if betas_df.columns.nlevels > 1 else betas_df.columns
betas_df = betas_df.stack().to_frame()
betas_df.columns = betas_df.columns if len(betas_df.columns) > 1 else [betas_df.columns[0]]

print("Calculando betas rolantes (isso pode levar alguns minutos)...")
print("Concluido.")

beta_long = pd.DataFrame()
for beta_name, beta_wide in betas.items():
    stacked = beta_wide.stack()
    stacked.name = beta_name
    stacked.index.names = ["date", "ticker"]
    if beta_long.empty:
        beta_long = stacked.to_frame()
    else:
        beta_long = beta_long.join(stacked, how="outer")

df = df.join(beta_long, how="left")

print(f"\nBetas adicionados. Shape do df: {df.shape}")
print(df.filter(like="beta_").describe().round(4))

### 2.7 Rankings cross-sectional
Para cada feature, calculamos o ranking percentil de cada acao em relacao
as demais no mesmo dia. Isso normaliza as features e permite comparacoes
justas entre acoes com escalas de preco diferentes.

In [ ]:
features_to_rank = [
    "ret_1d", "ret_5d", "ret_21d", "ret_63d",
    "volatility_gk_21d", "volatility_std_21d",
    "rsi_14", "macd_hist", "bb_width", "atr_14",
    "volume_ratio_21d", "obv_pct_change_21d",
    "beta_mkt_rf", "beta_smb", "beta_hml"
]

for feat in features_to_rank:
    if feat in df.columns:
        df[f"{feat}_rank"] = df.groupby(level="date")[feat].rank(pct=True)

print(f"Features de ranking adicionadas: {len([c for c in df.columns if c.endswith('_rank')])}")
print(f"Shape final: {df.shape}")

### 2.8 Filtro de liquidez
Mantemos apenas as 150 acoes mais liquidas em cada mes.
Acoes iliquidas geram sinais pouco confiaveis e sao caras pra operar.

In [ ]:
df["year_month"] = df.index.get_level_values("date").to_period("M")

monthly_dollar_vol = df.groupby(["year_month", "ticker"])["dollar_volume"].mean()
top_150 = monthly_dollar_vol.groupby("year_month").nlargest(150).reset_index(level=0, drop=True)
top_150_idx = top_150.index

df["is_liquid"] = df.set_index("year_month", append=True).index.droplevel("date").isin(top_150_idx)
df_filtered = df[df["is_liquid"]].copy()
df_filtered = df_filtered.drop(columns=["is_liquid", "year_month"])

print(f"Antes do filtro: {df.index.get_level_values('ticker').nunique()} tickers")
print(f"Depois do filtro: {df_filtered.index.get_level_values('ticker').nunique()} tickers unicos")
print(f"Shape filtrado: {df_filtered.shape}")

### 2.9 Limpeza e salvamento
Removemos linhas com NaN nas features principais (causados pelas janelas rolantes)
e salvamos o dataset processado.

In [ ]:
core_features = [
    "ret_1d", "ret_5d", "ret_21d", "ret_63d",
    "volatility_gk_21d", "volatility_std_21d",
    "rsi_14", "macd_hist", "bb_width", "atr_14",
    "volume_ratio_21d",
    "beta_mkt_rf",
    "target_21d"
]

df_clean = df_filtered.dropna(subset=core_features)

print(f"Antes da limpeza: {df_filtered.shape}")
print(f"Depois da limpeza: {df_clean.shape}")
print(f"Periodo: {df_clean.index.get_level_values('date').min()} ate {df_clean.index.get_level_values('date').max()}")
print(f"Tickers: {df_clean.index.get_level_values('ticker').nunique()}")
print(f"\nColunas ({len(df_clean.columns)}):")
print(df_clean.columns.tolist())

In [ ]:
df_clean.to_parquet("../data/processed/features.parquet")
print("Salvo em data/processed/features.parquet")
print(f"Tamanho: {df_clean.shape[0]:,} linhas x {df_clean.shape[1]} colunas")